In [1]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

df = pd.read_parquet(
    "../Data/processed/imd_temperature_2010_2025.parquet"
)

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["latitude", "longitude", "date"]
).reset_index(drop=True)

print(df.shape)

(2074620, 4)


In [2]:
group_cols = ["latitude", "longitude"]

df["month"] = df["date"].dt.month
df["day_of_year"] = df["date"].dt.dayofyear

df["month_sin"] = np.sin(
    2 * np.pi * df["day_of_year"] / 365.25
)

df["month_cos"] = np.cos(
    2 * np.pi * df["day_of_year"] / 365.25
)

df["temp_lag_1"] = (
    df.groupby(group_cols)["max_temperature"].shift(1)
)

df["temp_lag_2"] = (
    df.groupby(group_cols)["max_temperature"].shift(2)
)

df["temp_lag_3"] = (
    df.groupby(group_cols)["max_temperature"].shift(3)
)

df["temp_lag_7"] = (
    df.groupby(group_cols)["max_temperature"].shift(7)
)

df["temp_rolling_mean_3"] = (
    df.groupby(group_cols)["max_temperature"]
      .transform(lambda x: x.shift(1).rolling(3).mean())
)

df["temp_rolling_mean_7"] = (
    df.groupby(group_cols)["max_temperature"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

df["temp_rolling_std_7"] = (
    df.groupby(group_cols)["max_temperature"]
      .transform(lambda x: x.shift(1).rolling(7).std())
)

df["temperature_change_1d"] = (
    df["temp_lag_1"] - df["temp_lag_2"]
)

df["temperature_change_3d"] = (
    df["temp_lag_1"] - df["temp_lag_3"]
)

In [3]:
df["target_temperature"] = (
    df.groupby(group_cols)["max_temperature"].shift(-1)
)

In [4]:
train_mask = df["date"].dt.year <= 2022
val_mask   = df["date"].dt.year == 2023
test_mask  = df["date"].dt.year >= 2024

train_raw = df.loc[train_mask].copy()

In [5]:
thresholds = (
    train_raw.groupby(group_cols)["max_temperature"]
    .quantile([0.90, 0.95, 0.975, 0.99])
    .unstack()
    .reset_index()
)

thresholds.columns = [
    "latitude",
    "longitude",
    "p90_temperature",
    "p95_temperature",
    "p975_temperature",
    "p99_temperature"
]

df = df.merge(
    thresholds,
    on=group_cols,
    how="left"
)

In [9]:
thresholds = (
    train_raw.groupby(group_cols)["max_temperature"]
    .quantile([0.90, 0.95, 0.975, 0.99])
    .unstack()
    .reset_index()
)

thresholds.columns = [
    "latitude",
    "longitude",
    "p90_temperature",
    "p95_temperature",
    "p975_temperature",
    "p99_temperature"
]

df = df.merge(
    thresholds,
    on=group_cols,
    how="left"
)

In [10]:
monthly_baseline = (
    train_raw.groupby(
        ["latitude", "longitude", "month"]
    )["max_temperature"]
    .mean()
    .reset_index(name="monthly_baseline")
)

df = df.merge(
    monthly_baseline,
    on=["latitude", "longitude", "month"],
    how="left"
)

df["temperature_anomaly"] = (
    df["max_temperature"] - df["monthly_baseline"]
)

df["anomaly_lag_1"] = (
    df["temp_lag_1"] - df["monthly_baseline"]
)

KeyError: 'monthly_baseline'

In [11]:
df["extreme_heat"] = (
    df["max_temperature"] > df["p95_temperature"]
)

df["not_extreme"] = (
    ~df["extreme_heat"]
).astype(int)

df["streak_group"] = (
    df.groupby(group_cols)["not_extreme"].cumsum()
)

df["extreme_heat_streak"] = (
    df.groupby(
        group_cols + ["streak_group"]
    )["extreme_heat"]
    .transform("sum")
)

df.loc[
    ~df["extreme_heat"],
    "extreme_heat_streak"
] = 0

In [12]:
regression_features = [
    "latitude",
    "longitude",
    "month",
    "day_of_year",
    "month_sin",
    "month_cos",

    "max_temperature",

    "temp_lag_1",
    "temp_lag_2",
    "temp_lag_3",
    "temp_lag_7",

    "temp_rolling_mean_3",
    "temp_rolling_mean_7",
    "temp_rolling_std_7",

    "temperature_change_1d",
    "temperature_change_3d",

    "temperature_anomaly",
    "anomaly_lag_1",

    "extreme_heat_streak",

    "p90_temperature",
    "p95_temperature",
    "p975_temperature",
    "p99_temperature"
]

regression_df = df[
    ["date"] +
    regression_features +
    ["target_temperature"]
].dropna(
    subset=regression_features + ["target_temperature"]
).copy()

print(regression_df.shape)

(2055334, 25)


In [13]:
train_mask = regression_df["date"].dt.year <= 2022
val_mask   = regression_df["date"].dt.year == 2023
test_mask  = regression_df["date"].dt.year >= 2024

X_train = regression_df.loc[
    train_mask, regression_features
]
y_train = regression_df.loc[
    train_mask, "target_temperature"
]

X_val = regression_df.loc[
    val_mask, regression_features
]
y_val = regression_df.loc[
    val_mask, "target_temperature"
]

X_test = regression_df.loc[
    test_mask, regression_features
]
y_test = regression_df.loc[
    test_mask, "target_temperature"
]

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(1668083, 23)
(129138, 23)
(258113, 23)


In [14]:
lgb_reg = lgb.LGBMRegressor(
    objective="regression",

    n_estimators=2000,
    learning_rate=0.03,

    num_leaves=63,
    max_depth=-1,

    min_child_samples=50,

    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,

    reg_alpha=0.1,
    reg_lambda=0.5,

    random_state=42,
    n_jobs=-1
)

lgb_reg.fit(
    X_train,
    y_train,

    eval_set=[
        (X_train, y_train),
        (X_val, y_val)
    ],

    eval_names=[
        "train",
        "validation"
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            verbose=True
        ),
        lgb.log_evaluation(100)
    ]
)

c:\Users\Qudsiya Siddique\Desktop\Climat\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014965 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4906
[LightGBM] [Info] Number of data points in the train set: 1668083, number of used features: 23
[LightGBM] [Info] Start training from score 30.975245
Training until validation scores don't improve for 100 rounds
[100]	train's l2: 1.57498	validation's l2: 1.71517
[200]	train's l2: 1.41494	validation's l2: 1.66302
Early stopping, best iteration is:
[165]	train's l2: 1.44259	validation's l2: 1.65684


,num_leaves,63
,learning_rate,0.03
,n_estimators,2000
,objective,'regression'
,min_child_samples,50
,subsample,0.85
,subsample_freq,1
,colsample_bytree,0.85
,reg_alpha,0.1
,reg_lambda,0.5
,random_state,42


In [15]:
val_pred = lgb_reg.predict(
    X_val,
    num_iteration=lgb_reg.best_iteration_
)

test_pred = lgb_reg.predict(
    X_test,
    num_iteration=lgb_reg.best_iteration_
)

print("VALIDATION")
print("MAE :", mean_absolute_error(y_val, val_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_val, val_pred)))
print("R²  :", r2_score(y_val, val_pred))

print("\nTEST")
print("MAE :", mean_absolute_error(y_test, test_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, test_pred)))
print("R²  :", r2_score(y_test, test_pred))

VALIDATION
MAE : 0.9001261447828316
RMSE: 1.2871847986035239
R²  : 0.9465186391513533

TEST
MAE : 0.9161376527568175
RMSE: 1.2728193270336619
R²  : 0.9486192891471396


In [16]:
test_results = regression_df.loc[
    test_mask,
    [
        "date",
        "p95_temperature",
        "p975_temperature",
        "p99_temperature",
        "target_temperature"
    ]
].copy()

test_results["predicted_temperature"] = test_pred

In [17]:
test_results["predicted_severity"] = np.select(
    [
        test_results["predicted_temperature"] < test_results["p95_temperature"],

        (
            (test_results["predicted_temperature"] >= test_results["p95_temperature"]) &
            (test_results["predicted_temperature"] < test_results["p975_temperature"])
        ),

        (
            (test_results["predicted_temperature"] >= test_results["p975_temperature"]) &
            (test_results["predicted_temperature"] < test_results["p99_temperature"])
        )
    ],
    [
        0,
        1,
        2
    ],
    default=3
)

In [18]:
test_results["actual_severity"] = np.select(
    [
        test_results["target_temperature"] < test_results["p95_temperature"],

        (
            (test_results["target_temperature"] >= test_results["p95_temperature"]) &
            (test_results["target_temperature"] < test_results["p975_temperature"])
        ),

        (
            (test_results["target_temperature"] >= test_results["p975_temperature"]) &
            (test_results["target_temperature"] < test_results["p99_temperature"])
        )
    ],
    [
        0,
        1,
        2
    ],
    default=3
)

In [19]:

severity_eval = test_results[
    test_results["actual_severity"] > 0
].copy()

In [20]:
from sklearn.metrics import classification_report, confusion_matrix

print(
    classification_report(
        severity_eval["actual_severity"],
        severity_eval["predicted_severity"],
        labels=[1, 2, 3],
        target_names=[
            "Moderate",
            "High",
            "Extreme"
        ]
    )
)

print(
    confusion_matrix(
        severity_eval["actual_severity"],
        severity_eval["predicted_severity"],
        labels=[1, 2, 3]
    )
)

              precision    recall  f1-score   support

    Moderate       0.45      0.28      0.34      5479
        High       0.35      0.27      0.30      3605
     Extreme       0.82      0.44      0.57      3955

   micro avg       0.51      0.33      0.40     13039
   macro avg       0.54      0.33      0.41     13039
weighted avg       0.53      0.33      0.40     13039

[[1523  462  108]
 [1324  976  287]
 [ 542 1360 1753]]


In [21]:
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    lgb_reg,
    "../models/severity_lgbm.joblib"
)

['../models/severity_lgbm.joblib']

In [22]:
joblib.dump(
    {
        "features": regression_features,
        "p95": thresholds,
        "monthly_baseline": monthly_baseline,
        "best_iteration": lgb_reg.best_iteration_
    },
    "../models/severity_metadata.joblib"
)

['../models/severity_metadata.joblib']